# Feature engineering for the reference station CSVs

This notebook implements the engineered features 

1. Basic derived variables: wind_speed, wind direction sin/cos, relative humidity, cyclic time encodings.
2. Neighbour readings `nb_1` to `nb_5` (the 5 nearest reference stations) and their inverse distance weighted mean `idw_neighbour`.
3. Lagged block from the nearest neighbour: values 1, 3, 6 and 24 hours earlier plus a trailing 24 hour mean and std.
4. Wind block: distance weighted upwind and downwind neighbour averages (120 degree cones).
5. Convection diffusion terms: gradient, advection and diffusion from a quadratic surface fitted to the 8 nearest neighbours.
6. LCS features: `f_rh` hygroscopic growth, `lcs_x_frh`, `lcs_rh_corrected`, and the cluster advection `lcs_adv`.
7. Boundary layer features: `ventilation_coef`, `low_blh`, `stagnant` and `stagnation_hours`.
8. Fill plus indicator: missing engineered values are filled with the training years median, and columns missing more than 1% of their hours get a 0/1 `_isna` indicator column. This makes the table complete, so XGBoost and the AGNN receive identical input.


## 1. Settings and paths

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = r"C:\Users\Storm Anderson\Documents\UVA\Inferencing\data_imputed"
COORDS_CSV = r"C:\Users\Storm Anderson\Documents\UVA\Inferencing\station_cell_map.csv"
LCS_META_CSV = r"C:\Users\Storm Anderson\Documents\UVA\Inferencing\cleaned\lcs_meta.csv"

K_NEAREST = 5           # neighbours used for nb_1 to nb_5
K_QUAD = 8              # neighbours used for the quadratic surface fit
IDW_POWER = 2.0         # inverse distance weighting power
LAG_HOURS = [1, 3, 6, 24]
KAPPA = 0.5             # hygroscopic growth parameter in f(RH)
RH_CAP = 0.95           # cap the RH fraction so f(RH) does not blow up near saturation
CONE_HALF_DEG = 60.0    # half opening angle of the upwind/downwind cones (120 deg full cone)
WIND_THRESH = 3.2       # m/s, below this the wind counts as stagnant
PRECIP_THRESH = 1e-4    # m per hour, roughly 0.1 mm of rain
BLH_THRESH = 500.0      # m, shallow boundary layer threshold for stagnation
LOW_BLH_THRESH = 250.0  # m, very shallow boundary layer flag

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)

## 2. Load the imputed station data

In [ ]:
station_folders = sorted(os.listdir(DATA_DIR))

stations = {}
for folder in station_folders:
    folder_path = os.path.join(DATA_DIR, folder)
    if not os.path.isdir(folder_path):
        continue

    csv_path = os.path.join(folder_path, folder + ".csv")
    df = pd.read_csv(csv_path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)
    stations[folder] = df
    print(folder, "->", len(df), "rows")

print()
print("loaded", len(stations), "stations")

NL01485_PM2.5_lucht -> 35064 rows
NL01487_PM2.5_lucht -> 35064 rows
NL01488_PM2.5_lucht -> 35064 rows
NL01489_PM2.5_lucht -> 35064 rows
NL01491_PM2.5_lucht -> 35064 rows
NL01494_PM2.5_lucht -> 35064 rows
NL01496_PM2.5_lucht -> 35064 rows
NL01912_PM2.5_lucht -> 35064 rows
NL10136_PM2.5_lucht -> 35064 rows
NL10418_PM2.5_lucht -> 35064 rows
NL10449_PM2.5_lucht -> 35064 rows
NL10636_PM2.5_lucht -> 35064 rows
NL10643_PM2.5_lucht -> 35064 rows
NL10738_PM2.5_lucht -> 35064 rows
NL10741_PM2.5_lucht -> 35064 rows
NL10742_PM2.5_lucht -> 35064 rows
NL10821_PM2.5_lucht -> 35064 rows
NL10937_PM2.5_lucht -> 35064 rows
NL10938_PM2.5_lucht -> 35064 rows
NL49007_PM2.5_lucht -> 35064 rows
NL49012_PM2.5_lucht -> 35064 rows
NL49014_PM2.5_lucht -> 35064 rows
NL49016_PM2.5_lucht -> 35064 rows
NL49017_PM2.5_lucht -> 35064 rows
NL49551_PM2.5_lucht -> 35064 rows
NL49553_PM2.5_lucht -> 35064 rows
NL49557_PM2.5_lucht -> 35064 rows
NL49570_PM2.5_lucht -> 35064 rows
NL49572_PM2.5_lucht -> 35064 rows
NL49573_PM2.5_

## 3. Station coordinates and neighbour geometry

The station lat/lon come from `station_cell_map.csv`. For every station we sort all other
stations by haversine distance and keep their distance and compass bearing. The first 5 are used
for the neighbour readings and the wind block, the first 8 for the quadratic surface fit.

In [20]:
def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2.0) ** 2
    return 2.0 * R * np.arcsin(np.sqrt(a))


def bearing_rad(lat1, lon1, lat2, lon2):
    # compass bearing from point 1 to point 2, in radians from north, clockwise
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dlon = np.radians(lon2 - lon1)
    x = np.sin(dlon) * np.cos(phi2)
    y = np.cos(phi1) * np.sin(phi2) - np.sin(phi1) * np.cos(phi2) * np.cos(dlon)
    return np.mod(np.arctan2(x, y), 2.0 * np.pi)


def angle_diff(a, b):
    # shortest signed angle a minus b, in [-pi, pi]
    return np.mod(a - b + np.pi, 2.0 * np.pi) - np.pi


coords = pd.read_csv(COORDS_CSV)
coords = coords[coords["station_id"].isin(stations.keys())]
coords = coords.set_index("station_id")[["lat", "lon"]]
print("stations with coordinates:", len(coords), "of", len(stations))

neighbours = {}
for name in stations:
    lat0 = coords.loc[name, "lat"]
    lon0 = coords.loc[name, "lon"]
    rows = []
    for other in stations:
        if other == name:
            continue
        lat1 = coords.loc[other, "lat"]
        lon1 = coords.loc[other, "lon"]
        rows.append({
            "id": other,
            "lat": lat1,
            "lon": lon1,
            "dist_km": float(haversine_km(lon0, lat0, lon1, lat1)),
            "bearing_rad": float(bearing_rad(lat0, lon0, lat1, lon1)),
        })
    rows.sort(key=lambda r: r["dist_km"])
    neighbours[name] = rows

example = list(stations.keys())[0]
print(example, "nearest 5:")
for nb in neighbours[example][:5]:
    print("   ", nb["id"], "-", round(nb["dist_km"], 1), "km")

stations with coordinates: 31 of 31
NL01485_PM2.5_lucht nearest 5:
    NL10449_PM2.5_lucht - 5.6 km
    NL01494_PM2.5_lucht - 6.8 km
    NL01487_PM2.5_lucht - 9.0 km
    NL01491_PM2.5_lucht - 9.5 km
    NL01488_PM2.5_lucht - 9.5 km


## 4. Wide panel of reference PM2.5

Rows are hours, columns are stations, values are the ref_pm25. Every neighbour
feature below reads from this panel, a neighbour value is simply that neighbour's column.

In [21]:
panel = pd.DataFrame()
for name in stations:
    df = stations[name]
    panel[name] = pd.Series(df["ref_pm25"].values, index=df["datetime"])

panel = panel.sort_index()
print("panel shape:", panel.shape, "(hours x stations)")

panel shape: (35064, 31) (hours x stations)


## 5. Basic derived variables (wind, humidity, cyclic time)

Wind direction is stored as sin and cos of the direction the wind blows towards, so
u10 = wind_speed * wind_dir_sin and v10 = wind_speed * wind_dir_cos. Relative humidity
comes from t2m and d2m (both degC) with the Magnus formula.

In [22]:
for name in stations:
    df = stations[name]

    # wind speed and direction
    ws = np.sqrt(df["u10"].values ** 2 + df["v10"].values ** 2)
    ws_safe = np.where(ws > 0, ws, 1.0)
    df["wind_speed"] = ws
    df["wind_dir_sin"] = np.where(ws > 0, df["u10"].values / ws_safe, 0.0)
    df["wind_dir_cos"] = np.where(ws > 0, df["v10"].values / ws_safe, 0.0)

    # relative humidity in percent, Magnus formula over water
    es_t = 6.112 * np.exp(17.62 * df["t2m"].values / (243.12 + df["t2m"].values))
    es_d = 6.112 * np.exp(17.62 * df["d2m"].values / (243.12 + df["d2m"].values))
    df["rh"] = np.clip(100.0 * es_d / es_t, 0.0, 100.0)

    # cyclic time encodings
    doy = df["datetime"].dt.dayofyear.values
    df["hour_sin"] = np.sin(2.0 * np.pi * df["hour"].values / 24.0)
    df["hour_cos"] = np.cos(2.0 * np.pi * df["hour"].values / 24.0)
    df["doy_sin"] = np.sin(2.0 * np.pi * doy / 365.25)
    df["doy_cos"] = np.cos(2.0 * np.pi * doy / 365.25)
    df["dow_sin"] = np.sin(2.0 * np.pi * df["dayofweek"].values / 7.0)
    df["dow_cos"] = np.cos(2.0 * np.pi * df["dayofweek"].values / 7.0)

print("basic derived variables added")

basic derived variables added


## 6. Neighbour readings nb_1 to nb_5 and idw_neighbour

nb_i is the reference PM2.5 of the i-th nearest station at the same hour. idw_neighbour
combines the five with weights 1 / distance^2 (distances are floored at 0.1 km).

In [23]:
for name in stations:
    df = stations[name]
    meta = neighbours[name][:K_NEAREST]

    for i in range(K_NEAREST):
        nb_id = meta[i]["id"]
        df["nb_" + str(i + 1)] = panel[nb_id].reindex(df["datetime"]).values

    weights = []
    for i in range(K_NEAREST):
        d = max(meta[i]["dist_km"], 0.1)
        weights.append(1.0 / d ** IDW_POWER)

    num = np.zeros(len(df))
    den = np.zeros(len(df))
    for i in range(K_NEAREST):
        vals = df["nb_" + str(i + 1)].values
        ok = ~np.isnan(vals)
        num = num + np.where(ok, vals * weights[i], 0.0)
        den = den + np.where(ok, weights[i], 0.0)
    df["idw_neighbour"] = np.where(den > 0, num / den, np.nan)

print("neighbour readings and idw_neighbour added")

neighbour readings and idw_neighbour added


## 7. Lagged block from the nearest neighbour

From the nearest neighbour's series: the value 1, 3, 6 and 24 hours earlier, plus a trailing
24 hour mean and std. The rolling window is left closed, so the current hour never enters
its own window and no future value is used.

In [24]:
for name in stations:
    df = stations[name]
    nb1_id = neighbours[name][0]["id"]
    nb1 = panel[nb1_id]

    for h in LAG_HOURS:
        shifted = nb1.reindex(df["datetime"] - pd.Timedelta(hours=h))
        df["nb1_lag_" + str(h) + "h"] = shifted.values

    roll_mean = nb1.rolling("24h", closed="left", min_periods=12).mean()
    roll_std = nb1.rolling("24h", closed="left", min_periods=12).std()
    df["nb1_24h_mean"] = roll_mean.reindex(df["datetime"]).values
    df["nb1_24h_std"] = roll_std.reindex(df["datetime"]).values

print("lagged block added")

lagged block added


## 8. Wind block: upwind and downwind neighbour averages

The wind vector points towards theta_to = arctan2(u10, v10), the air comes from
theta_from = theta_to + pi. A neighbour is upwind when its bearing lies within 60 degrees
of theta_from, downwind when it lies within 60 degrees of theta_to. Neighbour values in
each cone are averaged with weight 1 / distance, and the number of neighbours in each cone
is kept as a count.

In [25]:
cone_half_rad = np.radians(CONE_HALF_DEG)

for name in stations:
    df = stations[name]
    meta = neighbours[name][:K_NEAREST]

    theta_to = np.mod(np.arctan2(df["u10"].values, df["v10"].values), 2.0 * np.pi)
    theta_from = np.mod(theta_to + np.pi, 2.0 * np.pi)

    n = len(df)
    up_num = np.zeros(n)
    up_den = np.zeros(n)
    up_count = np.zeros(n, dtype=int)
    dn_num = np.zeros(n)
    dn_den = np.zeros(n)
    dn_count = np.zeros(n, dtype=int)

    for i in range(K_NEAREST):
        pm = df["nb_" + str(i + 1)].values
        valid = ~np.isnan(pm)
        w = 1.0 / max(meta[i]["dist_km"], 0.1)
        beta = meta[i]["bearing_rad"]

        in_up = (np.abs(angle_diff(beta, theta_from)) <= cone_half_rad) & valid
        in_dn = (np.abs(angle_diff(beta, theta_to)) <= cone_half_rad) & valid

        up_num = up_num + np.where(in_up, pm * w, 0.0)
        up_den = up_den + np.where(in_up, w, 0.0)
        up_count = up_count + in_up.astype(int)
        dn_num = dn_num + np.where(in_dn, pm * w, 0.0)
        dn_den = dn_den + np.where(in_dn, w, 0.0)
        dn_count = dn_count + in_dn.astype(int)

    df["pm_upwind"] = np.where(up_den > 0, up_num / up_den, np.nan)
    df["pm_downwind"] = np.where(dn_den > 0, dn_num / dn_den, np.nan)
    df["n_upwind"] = up_count
    df["n_downwind"] = dn_count

print("wind block added")

C:\Users\Storm Anderson\AppData\Local\Temp\ipykernel_18876\3796866764.py:34: RuntimeWarning: invalid value encountered in divide
  df["pm_upwind"] = np.where(up_den > 0, up_num / up_den, np.nan)
C:\Users\Storm Anderson\AppData\Local\Temp\ipykernel_18876\3796866764.py:35: RuntimeWarning: invalid value encountered in divide
  df["pm_downwind"] = np.where(dn_den > 0, dn_num / dn_den, np.nan)


wind block added


## 9. Convection diffusion terms from the 8 nearest neighbours

A local quadratic surface is fitted to the 8 nearest neighbours (least squares). 
Read off at the station itself (dx = dy = 0):

* pm_gradient = sqrt(b^2 + c^2), the size of the horizontal PM2.5 gradient, in ug/m3 per km
* pm_advection = -(u10 * b + v10 * c) * 3.6, wind carrying the gradient past the station, in ug/m3 per hour
* pm_diffusion = 2d + 2e, the Laplacian, in ug/m3 per km^2


In [26]:
for name in stations:
    df = stations[name]
    meta = neighbours[name][:K_QUAD]
    lat0 = coords.loc[name, "lat"]
    lon0 = coords.loc[name, "lon"]

    # neighbour positions in km relative to the station (flat map approximation)
    A = np.zeros((K_QUAD, 6))
    for i in range(K_QUAD):
        dx = (meta[i]["lon"] - lon0) * 111.32 * np.cos(np.radians(lat0))
        dy = (meta[i]["lat"] - lat0) * 110.57
        A[i, 0] = 1.0
        A[i, 1] = dx
        A[i, 2] = dy
        A[i, 3] = dx * dx
        A[i, 4] = dy * dy
        A[i, 5] = dx * dy

    A_pinv = np.linalg.pinv(A)

    values = np.zeros((len(df), K_QUAD))
    for i in range(K_QUAD):
        nb_id = meta[i]["id"]
        values[:, i] = panel[nb_id].reindex(df["datetime"]).values

    coef = values @ A_pinv.T
    bad = np.isnan(values).any(axis=1)
    coef[bad, :] = np.nan

    grad_x = coef[:, 1]
    grad_y = coef[:, 2]
    df["pm_gradient"] = np.sqrt(grad_x ** 2 + grad_y ** 2)
    df["pm_advection"] = -(df["u10"].values * grad_x + df["v10"].values * grad_y) * 3.6
    df["pm_diffusion"] = 2.0 * coef[:, 3] + 2.0 * coef[:, 4]

print("convection diffusion terms added")

convection diffusion terms added


## 10. LCS features: hygroscopic growth and cluster advection

lcs_x_frh is the interaction, lcs_rh_corrected the de-grown estimate. For the advection term the cluster
location is the nearest centroid from lcs_meta.csv, and the coefficient is (wind_speed / distance) * max(0, cos(bearing difference)). 
So the signal is strongest when the wind blows straight from the cluster towards the station.

In [ ]:
lcs_meta = pd.read_csv(LCS_META_CSV)

for name in stations:
    df = stations[name]

    # hygroscopic growth factor and interactions with the LCS median
    rh_frac = np.clip(df["rh"].values / 100.0, 0.0, RH_CAP)
    df["f_rh"] = 1.0 + KAPPA * rh_frac / (1.0 - rh_frac)
    df["lcs_x_frh"] = df["lcs_median_pm25"] * df["f_rh"]
    df["lcs_rh_corrected"] = df["lcs_median_pm25"] / df["f_rh"]

    # nearest LCS centroid
    lat0 = coords.loc[name, "lat"]
    lon0 = coords.loc[name, "lon"]
    dists = haversine_km(lon0, lat0, lcs_meta["lon"].values, lcs_meta["lat"].values)
    j = int(np.argmin(dists))
    city_lat = float(lcs_meta["lat"].iloc[j])
    city_lon = float(lcs_meta["lon"].iloc[j])
    city_km = max(float(dists[j]), 0.1)

    # directional advection coefficient, cluster towards station
    ws = df["wind_speed"].values
    theta_to = np.mod(np.arctan2(df["u10"].values, df["v10"].values), 2.0 * np.pi)
    beta = bearing_rad(city_lat, city_lon, lat0, lon0)
    align = np.maximum(0.0, np.cos(angle_diff(beta, theta_to)))
    df["lcs_adv_coef"] = (ws / city_km) * align
    df["lcs_adv"] = df["lcs_adv_coef"] * df["lcs_median_pm25"]

print("lcs features added")

lcs features added


## 11. Boundary layer features

An hour is stagnant when the wind is weak, there is hardly any rain and the 
boundary layer is shallow. stagnation_hours counts how many stagnant hours 
in a row the station has seen, and resets on a non stagnant hour.

In [28]:
for name in stations:
    df = stations[name]

    df["ventilation_coef"] = df["blh"] * df["wind_speed"]
    df["low_blh"] = (df["blh"] < LOW_BLH_THRESH).astype(int)

    stagnant = (df["wind_speed"] < WIND_THRESH) & (df["tp"] < PRECIP_THRESH) & (df["blh"] < BLH_THRESH)
    df["stagnant"] = stagnant.astype(int)

    s = df["stagnant"].values
    acc = np.zeros(len(s), dtype=int)
    run = 0
    for i in range(len(s)):
        if s[i] == 1:
            run = run + 1
        else:
            run = 0
        acc[i] = run
    df["stagnation_hours"] = acc

print("boundary layer features added")

boundary layer features added


## 12. Fill missing values plus indicator columns

XGBoost handles NaN natively but the AGNN cannot. Both models should see the exact same table. So every engineered column
is made complete here: missing values are replaced with the column median computed on the training years only 
(year <= FILL_YEAR_MAX), and any column missing more than 1% of its hours gets a companion 0/1 indicator column named 
<column>_isna that marks the filled rows. The main case is the wind block: pm_upwind is missing when no neighbour sits 
inside the upwind cone at that hour, which is real information, and the indicator (together with n_upwind being 0) keeps
that information visible after the fill.

In [ ]:
FILL_YEAR_MAX = 2022        # fill statistics come from these years only, so the test year cannot leak in
INDICATOR_MIN_SHARE = 0.01  # add an _isna indicator when more than 1% of the hours are missing

NEW_COLS = [
    "wind_speed", "wind_dir_sin", "wind_dir_cos", "rh",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos", "dow_sin", "dow_cos",
    "nb_1", "nb_2", "nb_3", "nb_4", "nb_5", "idw_neighbour",
    "nb1_lag_1h", "nb1_lag_3h", "nb1_lag_6h", "nb1_lag_24h", "nb1_24h_mean", "nb1_24h_std",
    "pm_upwind", "pm_downwind", "n_upwind", "n_downwind",
    "pm_gradient", "pm_advection", "pm_diffusion",
    "f_rh", "lcs_x_frh", "lcs_rh_corrected", "lcs_adv_coef", "lcs_adv",
    "ventilation_coef", "low_blh", "stagnant", "stagnation_hours",
]

# missing share per column, and the training years median as fill value
fill_values = {}
missing_share = {}
for col in NEW_COLS:
    train_parts = []
    all_parts = []
    for name in stations:
        df = stations[name]
        train_parts.append(df.loc[df["year"] <= FILL_YEAR_MAX, col])
        all_parts.append(df[col])
    train_s = pd.concat(train_parts)
    all_s = pd.concat(all_parts)
    fill_values[col] = float(train_s.median())
    missing_share[col] = float(all_s.isna().mean())

# which columns get an indicator
INDICATOR_COLS = []
for col in NEW_COLS:
    if missing_share[col] > INDICATOR_MIN_SHARE:
        INDICATOR_COLS.append(col + "_isna")

# add the indicators first, then fill every missing value
for name in stations:
    df = stations[name]
    for col in NEW_COLS:
        if missing_share[col] > INDICATOR_MIN_SHARE:
            df[col + "_isna"] = df[col].isna().astype(int)
        df[col] = df[col].fillna(fill_values[col])

print("columns that had missing values (share, fill value):")
for col in NEW_COLS:
    if missing_share[col] > 0:
        print("   %-18s %6.2f%%  -> filled with %.4g" % (col, 100.0 * missing_share[col], fill_values[col]))
print()
print("indicator columns added:", INDICATOR_COLS)

columns that had missing values (share, fill value):
   nb1_lag_1h           0.00%  -> filled with 7.96
   nb1_lag_3h           0.01%  -> filled with 7.955
   nb1_lag_6h           0.02%  -> filled with 7.955
   nb1_lag_24h          0.07%  -> filled with 7.975
   nb1_24h_mean         0.03%  -> filled with 8.387
   nb1_24h_std          0.03%  -> filled with 3.492
   pm_upwind           34.13%  -> filled with 7.63
   pm_downwind         33.78%  -> filled with 8.224

indicator columns added: ['pm_upwind_isna', 'pm_downwind_isna']


## 13. Save the CSVs back into data_imputed

In [30]:
for name in stations:
    df = stations[name]
    out_path = os.path.join(DATA_DIR, name, name + ".csv")
    df.to_csv(out_path, index=False)
    print(name, "-> saved with", df.shape[1], "columns")

print()
print("engineered columns:", len(NEW_COLS), "| indicator columns:", len(INDICATOR_COLS))

NL01485_PM2.5_lucht -> saved with 97 columns
NL01487_PM2.5_lucht -> saved with 97 columns
NL01488_PM2.5_lucht -> saved with 97 columns
NL01489_PM2.5_lucht -> saved with 97 columns
NL01491_PM2.5_lucht -> saved with 97 columns
NL01494_PM2.5_lucht -> saved with 97 columns
NL01496_PM2.5_lucht -> saved with 97 columns
NL01912_PM2.5_lucht -> saved with 97 columns
NL10136_PM2.5_lucht -> saved with 97 columns
NL10418_PM2.5_lucht -> saved with 97 columns
NL10449_PM2.5_lucht -> saved with 97 columns
NL10636_PM2.5_lucht -> saved with 97 columns
NL10643_PM2.5_lucht -> saved with 97 columns
NL10738_PM2.5_lucht -> saved with 97 columns
NL10741_PM2.5_lucht -> saved with 97 columns
NL10742_PM2.5_lucht -> saved with 97 columns
NL10821_PM2.5_lucht -> saved with 97 columns
NL10937_PM2.5_lucht -> saved with 97 columns
NL10938_PM2.5_lucht -> saved with 97 columns
NL49007_PM2.5_lucht -> saved with 97 columns
NL49012_PM2.5_lucht -> saved with 97 columns
NL49014_PM2.5_lucht -> saved with 97 columns
NL49016_PM

## 14. Value ranges of all new columns

Pooled over all stations, indicator columns included. 

In [31]:
REPORT_COLS = NEW_COLS + INDICATOR_COLS

rows = []
for col in REPORT_COLS:
    parts = []
    for name in stations:
        parts.append(stations[name][col])
    s = pd.concat(parts)

    rows.append({
        "column": col,
        "%_filled": round(float(s.notna().mean() * 100.0), 2),
        "min": "%.4g" % s.min(),
        "p1": "%.4g" % s.quantile(0.01),
        "p50": "%.4g" % s.quantile(0.50),
        "p99": "%.4g" % s.quantile(0.99),
        "max": "%.4g" % s.max(),
    })

ranges = pd.DataFrame(rows)
ranges

,column,%_filled,min,p1,p50,p99,max
0,wind_speed,100.0,0.02146,0.6939,3.909,11.29,19.58
1,wind_dir_sin,100.0,-1,-0.9988,0.3967,0.9997,1
2,wind_dir_cos,100.0,-1,-0.9987,0.2536,0.9996,1
3,rh,100.0,19.83,46.18,82.64,98.36,100
4,hour_sin,100.0,-1,-1,6.123e-17,1,1
5,hour_cos,100.0,-1,-1,-6.123e-17,1,1
6,doy_sin,100.0,-1,-0.9994,-0.004301,0.9996,1
7,doy_cos,100.0,-1,-0.9996,0.001075,0.9994,1
8,dow_sin,100.0,-0.9749,-0.9749,0,0.9749,0.9749
9,dow_cos,100.0,-0.901,-0.901,-0.2225,1,1
